<a href="https://colab.research.google.com/github/AkhileshSR/AkhileshSR/blob/main/102025_12_day_12_rag_with_pdfs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval-Augmented Generation (RAG) with PDFs

In [ ]:
!pip install openai faiss-cpu PyPDF2 pdfplumber nltk --quiet

In [ ]:
from openai import OpenAI
from getpass import getpass
import os
import numpy as np
import faiss
import pdfplumber
from PyPDF2 import PdfReader
import nltk
nltk.download('punkt')

# 🔑 Enter your OpenAI API Key
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
client = OpenAI()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Enter your OpenAI API key: ··········


# 2️⃣ Read a PDF using Two Libraries
Let's compare how PyPDF2 and pdfplumber extract text from the same document.

We will upload two pdfs for testing - simple_pdf and complex_pdf

In [ ]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
    simple_pdf = filename
    print(f"User uploaded file '{simple_pdf}'")

Saving Agentic-AI-transforming-Businesses.pdf to Agentic-AI-transforming-Businesses.pdf
User uploaded file 'Agentic-AI-transforming-Businesses.pdf'


In [ ]:
# --- PyPDF2 ---
reader = PdfReader(simple_pdf)
text_pypdf = ""
for page in reader.pages:
    text_pypdf += page.extract_text()
print("PyPDF2 Text Sample:\n", text_pypdf[:800])

PyPDF2 Text Sample:
 How is Agentic AI transforming 
businesses?
By Sridhar Jammalamadaka
Future Proof IndiaWhat is Generative AI?
You've used ChatGPT.com, Gemini.com, Perplexity .ai
What it is
AI that creates new content: text, 
images, code, audio, videoKey difference
Not just answering questions, it 
generates new things
The power
It can create, not just retrieve"The development of AI is as fundamental as the 
creation of the microprocessor, the personal 
computer, the Internet, and the mobile phone."
4 Bill GatesThe Big Picture: AI Evolution 
Framework
Complete Evolution Path
Perception AI (Past)
Systems that perceive and 
understand. Computer vision, 
speech recognition. Foundation 
for everything else.
Generative AI (Present)
Creating new content, text, images, 
code. This is what we're focusing on 
toda


In [ ]:
# --- pdfplumber ---
text_plumber = ""
with pdfplumber.open(simple_pdf) as pdf:
    for page in pdf.pages:
        text_plumber += page.extract_text()
print("pdfplumber Text Sample:\n", text_plumber[:800])

pdfplumber Text Sample:
 How is Agentic AI transforming
businesses?
By Sridhar Jammalamadaka
Future Proof IndiaWhat is Generative AI?
You've used ChatGPT.com, Gemini.com, Perplexity.ai
What it is Key difference
AI that creates new content: text, Not just answering questions, it
images, code, audio, video generates new things
The power
It can create, not just retrieve"The development of AI is as fundamental as the
creation of the microprocessor, the personal
computer, the Internet, and the mobile phone."
4 Bill GatesThe Big Picture: AI Evolution
Framework
Complete Evolution Path
Perception AI (Past)
Systems that perceive and
understand. Computer vision,
speech recognition. Foundation
for everything else.
Generative AI (Present)
Creating new content, text, images,
code. This is what we're focusing on
today
Agentic A


💬 Explanation

PyPDF2: good for simple, text-based PDFs; fast, but may merge words or skip layout spacing.

pdfplumber: more accurate spacing, preserves paragraph structure, can handle tables.

👉 Run this and visually compare the difference. You’ll see why extraction quality matters for downstream embeddings.

# ✂️ 3️⃣ Chunking Strategies

Let’s turn the text into chunks suitable for embedding.
We’ll try 3 chunking approaches and compare.

## A. Fixed-Size Chunking (Simple)

In [ ]:
def chunk_text_fixed(text, max_words=200):
    words = text.split()
    chunks = [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)]
    return chunks

chunks_fixed = chunk_text_fixed(text_plumber, max_words=10)
print("Fixed-size chunks:", len(chunks_fixed))
for chunk in chunks_fixed[:3]:
    print(chunk[:300])
    print("-" * 80)


Fixed-size chunks: 118
How is Agentic AI transforming businesses? By Sridhar Jammalamadaka Future
--------------------------------------------------------------------------------
Proof IndiaWhat is Generative AI? You've used ChatGPT.com, Gemini.com, Perplexity.ai
--------------------------------------------------------------------------------
What it is Key difference AI that creates new content:
--------------------------------------------------------------------------------


In [ ]:
chunks_fixed = chunk_text_fixed(text_plumber, 100)
print("Fixed-size chunks:", len(chunks_fixed))
for chunk in chunks_fixed[:3]:
    print(chunk[:300])
    print("-" * 80)

Fixed-size chunks: 12
How is Agentic AI transforming businesses? By Sridhar Jammalamadaka Future Proof IndiaWhat is Generative AI? You've used ChatGPT.com, Gemini.com, Perplexity.ai What it is Key difference AI that creates new content: text, Not just answering questions, it images, code, audio, video generates new thing
--------------------------------------------------------------------------------
AI (Present) Creating new content, text, images, code. This is what we're focusing on today Agentic AI (Emerging) Autonomous agents that reason, plan, and act. Goal-directed systems with persistent state. (Modules 3 & 4 will cover in depth.) Physical AI (Future) Autonomous systems, robotics, IoT con
--------------------------------------------------------------------------------
and act 1 2 3 4 5 2015s: Robotic Process 2022: AI Assistants + GenAI Automation (RPA) ChatGPT, natural language interfaces Automating repetitive tasks, UI automationBusiness Applications Across Key Functions Develo

✅ Pros: simple, consistent sizes

⚠️ Cons: breaks sentences or paragraphs

## B. Overlapping Chunking

Preserves continuity between chunks — useful for context consistency.

In [ ]:
def chunk_text_overlap(text, max_words=200, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words - overlap):
        chunk = " ".join(words[i:i + max_words])
        chunks.append(chunk)
    return chunks

chunks_overlap = chunk_text_overlap(text_plumber, 10, 4)
print("Overlapping chunks:", len(chunks_overlap))
for chunk in chunks_overlap[:3]:
    print(chunk[:300])
    print("-" * 80)

Overlapping chunks: 197
How is Agentic AI transforming businesses? By Sridhar Jammalamadaka Future
--------------------------------------------------------------------------------
By Sridhar Jammalamadaka Future Proof IndiaWhat is Generative AI? You've
--------------------------------------------------------------------------------
is Generative AI? You've used ChatGPT.com, Gemini.com, Perplexity.ai What it
--------------------------------------------------------------------------------


In [ ]:
chunks_overlap = chunk_text_overlap(text_plumber, 200, 30)
print("Overlapping chunks:", len(chunks_overlap))
for chunk in chunks_overlap[:3]:
    print(chunk[:300])
    print("-" * 80)

Overlapping chunks: 7
How is Agentic AI transforming businesses? By Sridhar Jammalamadaka Future Proof IndiaWhat is Generative AI? You've used ChatGPT.com, Gemini.com, Perplexity.ai What it is Key difference AI that creates new content: text, Not just answering questions, it images, code, audio, video generates new thing
--------------------------------------------------------------------------------
emergeFrom Simple Rules to Agentic AI Orchestration 2010s: Rule-based workflows 2019s: Workflow automation 2025: AI Agents Simple if-then logic, basic automation Connecting systems, API integrations Autonomous systems that reason, plan, and act 1 2 3 4 5 2015s: Robotic Process 2022: AI Assistants + 
--------------------------------------------------------------------------------
global social media posts markets Customer Service Document Processing Chatbots for customer support, automated response Document summarisation, content extraction and analysis generationEnterprise AI Capabilities 

✅ Pros: smoother retrieval, less context loss

⚠️ Cons: more total chunks (more embeddings = higher cost)

## C. Sentence-Based Chunking

Groups sentences instead of raw word counts — produces more semantically coherent pieces.

In [ ]:
from nltk import sent_tokenize
import nltk
nltk.download('punkt_tab')

def chunk_by_sentences(text, max_sentences=5):
    sentences = sent_tokenize(text)
    chunks = [" ".join(sentences[i:i+max_sentences]) for i in range(0, len(sentences), max_sentences)]
    return chunks

chunks_sentence = chunk_by_sentences(text_plumber, max_sentences=20)
print("Sentence-based chunks:", len(chunks_sentence))
for chunk in chunks_sentence[:3]:
    print(chunk)
    print("-" * 80)

Sentence-based chunks: 2
How is Agentic AI transforming
businesses? By Sridhar Jammalamadaka
Future Proof IndiaWhat is Generative AI? You've used ChatGPT.com, Gemini.com, Perplexity.ai
What it is Key difference
AI that creates new content: text, Not just answering questions, it
images, code, audio, video generates new things
The power
It can create, not just retrieve"The development of AI is as fundamental as the
creation of the microprocessor, the personal
computer, the Internet, and the mobile phone." 4 Bill GatesThe Big Picture: AI Evolution
Framework
Complete Evolution Path
Perception AI (Past)
Systems that perceive and
understand. Computer vision,
speech recognition. Foundation
for everything else. Generative AI (Present)
Creating new content, text, images,
code. This is what we're focusing on
today
Agentic AI (Emerging)
Autonomous agents that reason, plan, and
act. Goal-directed systems with persistent
state. (Modules 3 & 4 will cover in depth.) Physical AI (Future)
Autonomous sy

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
chunks_sentence = chunk_by_sentences(text_plumber, max_sentences=5)
print("Sentence-based chunks:", len(chunks_sentence))
for chunk in chunks_sentence[:3]:
    print(chunk)
    print("-" * 80)

Sentence-based chunks: 6
How is Agentic AI transforming
businesses? By Sridhar Jammalamadaka
Future Proof IndiaWhat is Generative AI? You've used ChatGPT.com, Gemini.com, Perplexity.ai
What it is Key difference
AI that creates new content: text, Not just answering questions, it
images, code, audio, video generates new things
The power
It can create, not just retrieve"The development of AI is as fundamental as the
creation of the microprocessor, the personal
computer, the Internet, and the mobile phone." 4 Bill GatesThe Big Picture: AI Evolution
Framework
Complete Evolution Path
Perception AI (Past)
Systems that perceive and
understand. Computer vision,
speech recognition.
--------------------------------------------------------------------------------
Foundation
for everything else. Generative AI (Present)
Creating new content, text, images,
code. This is what we're focusing on
today
Agentic AI (Emerging)
Autonomous agents that reason, plan, and
act. Goal-directed systems with persiste

✅ Pros: natural breaks in meaning

⚠️ Cons: variable chunk lengths; not ideal for every embedding model

# 🧮 4️⃣ Choose the Best Chunking Strategy

For most documents:
👉 Overlapping chunking gives the best tradeoff between context and structure.

Let’s proceed using that.

In [ ]:
chunks = chunks_overlap  # you can swap in other methods to compare (chunks_fixed / chunks_overlap / chunks_sentence)
for chunk in chunks[:3]:
    print(chunk[:300])
    print("-" * 80)

How is Agentic AI transforming businesses? By Sridhar Jammalamadaka Future Proof IndiaWhat is Generative AI? You've used ChatGPT.com, Gemini.com, Perplexity.ai What it is Key difference AI that creates new content: text, Not just answering questions, it images, code, audio, video generates new thing
--------------------------------------------------------------------------------
emergeFrom Simple Rules to Agentic AI Orchestration 2010s: Rule-based workflows 2019s: Workflow automation 2025: AI Agents Simple if-then logic, basic automation Connecting systems, API integrations Autonomous systems that reason, plan, and act 1 2 3 4 5 2015s: Robotic Process 2022: AI Assistants + 
--------------------------------------------------------------------------------
global social media posts markets Customer Service Document Processing Chatbots for customer support, automated response Document summarisation, content extraction and analysis generationEnterprise AI Capabilities - What Can AI Do? 01 0

# 🔢 5️⃣ Create Embeddings

Convert chunks into vectors.

In [ ]:
embeddings = []
for chunk in chunks:
    emb = client.embeddings.create(
        input=chunk,
        model="text-embedding-3-small"
    ).data[0].embedding
    embeddings.append(emb)

embedding_matrix = np.array(embeddings).astype("float32")
print("✅ Created embeddings for", len(chunks), "chunks.")

✅ Created embeddings for 7 chunks.


# 🧱 6️⃣ Build a Vector Database (FAISS)

In [ ]:
index = faiss.IndexFlatL2(embedding_matrix.shape[1])
index.add(embedding_matrix)
print(f"✅ Added {index.ntotal} vectors to FAISS index.")

✅ Added 7 vectors to FAISS index.


# 🔍 7️⃣ Define Retrieval

Find chunks most similar to a query.

In [ ]:
def retrieve(query, k=3):
    query_emb = client.embeddings.create(
        input=query,
        model="text-embedding-3-small"
    ).data[0].embedding
    query_vector = np.array([query_emb]).astype("float32")
    D, I = index.search(query_vector, k)
    return [(chunks[i], D[0][j]) for j, i in enumerate(I[0])]

# 🤖 8️⃣ Retrieval-Augmented Generation (RAG)

Combine retrieved context with a question to generate an informed answer.

In [ ]:
def rag_answer(query, k=3):
    results = retrieve(query, k)
    context = "\n".join([r[0] for r in results])

    prompt = f"""
    Use the context below to answer the question truthfully.
    If you cannot find the answer in context, say "I don’t know."

    Context:
    {context}

    Question: {query}
    Answer:
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()

# 💬 9️⃣ Try It Out

Ask real questions from your PDF.

In [ ]:
queries = [
    "What does the document say about real estate deep agents?",
    "What are the features?",
    "What is the technologies used?",
    "Summarize the main goal of this document."
]

for q in queries:
    print(f"Q: {q}")
    print("A:", rag_answer(q))
    print("-" * 80)

Q: What does the document say about real estate deep agents?
A: I don’t know.
--------------------------------------------------------------------------------
Q: What are the features?
A: The features include:

1. **Response Prompts**: Enhances results by generating better prompts.
2. **LLM Models Overview**: Information on various LLM models such as GPT Series, Claude Series, and Gemini Series, highlighting their strengths and intended uses.
3. **Latest Releases**: Updates on new LLM models and their capabilities along with release dates.
4. **Limitations of Basic LLMs**: Acknowledgment of limitations like lack of access to private knowledge, inability to perform real-time actions, and interactivity.
5. **Beyond LLMs**: Introduction to advanced AI like RAG (Retrieval-Augmented Generation) and Agentic AI, which provide access to documents and the ability to take actions.
6. **Practical Tools for Productivity**: Tools designed to solve real problems and improve workflows.
7. **AI Tools 

# 🧪 10 Reflection — Experiment

Try:

Different chunk sizes (100, 300, 500 words).

Overlap values (0, 50, 100).

Compare FAISS search results using different queries.